# Agentic Retrieval with RAGNav (Mistral)

This cookbook mirrors the *"agentic retrieval"* idea popularized by PageIndex: instead of treating retrieval as a fixed pipeline, you can **prompt an agent** to retrieve evidence on-demand.

**What RAGNav improves vs tree-only approaches**:
- **Hybrid-first** retrieval (BM25 + embeddings) to avoid LLM-only search bottlenecks
- **Multi-doc native** routing + filtering
- **Agent-friendly** retrieval primitive: `retrieve_raw()` returns structured, citeable snippets

> Requirements: `pip install -e ".[mistral]"` and `MISTRAL_API_KEY` set in your environment.


## Step 0: Install + set keys

```bash
pip install -e .
pip install -e ".[mistral]"
export MISTRAL_API_KEY="..."
```

If you prefer `.env`, copy `.env.example` to `.env` (never commit real keys).


In [ ]:
from ragnav.env import load_env
from ragnav.ingest.markdown import ingest_markdown_string
from ragnav.llm.mistral import MistralClient
from ragnav.retrieval import RAGNavIndex, RAGNavRetriever
from ragnav.display import print_wrapped

# Load `.env` if present (recommended for local dev)
load_env()

llm = MistralClient()


## Step 1: Ingest a small multi-doc corpus

RAGNav’s ingestion normalizes content into **blocks** (sections/headings/etc.).
For this minimal demo, we use Markdown strings.


In [ ]:
DOC_SURVEY = """
# Context Engineering Survey (Excerpt)

## 6. Evaluation
### 6.1 Evaluation Frameworks and Methodologies
Component-level assessment includes prompt evaluation, long-context evaluation (needle-in-a-haystack),
self-refinement, and structured/relational data integration.

### 6.2 Benchmarks
Benchmarks include LongMemEval for memory, BFCL/T-Eval for tool use, and WebArena for web agents.
"""

DOC_FINANCE = """
# ACME Corp Q1 FY25 Earnings (Excerpt)

## EBITDA Adjustments
We adjusted EBITDA for stock-based compensation and restructuring costs.
"""

docs = []
blocks = []
for name, content, meta in [
    ("survey.md", DOC_SURVEY, {"type": "paper", "topic": "eval"}),
    ("finance.md", DOC_FINANCE, {"type": "finance", "topic": "earnings"}),
]:
    d, b = ingest_markdown_string(content, name=name, metadata=meta)
    docs.append(d)
    blocks.extend(b)

index = RAGNavIndex.build(documents=docs, blocks=blocks, llm=llm)
retriever = RAGNavRetriever(index=index, llm=llm)
print(f"docs={len(docs)}, blocks={len(blocks)}")


## Step 2: Expose an agent-native retrieval tool

PageIndex’s cookbook uses a prompt that asks the system to return *raw relevant content*.
RAGNav provides `retrieve_raw()` for exactly this: a **JSON-serializable list** with stable `doc_id`/`block_id` and anchors.


In [ ]:
import json

def ragnav_retrieve_raw(query: str, max_blocks: int = 6):
    return retriever.retrieve_raw(query, max_blocks=max_blocks)

hits = ragnav_retrieve_raw("evaluation methods", max_blocks=4)
print(json.dumps(hits, indent=2)[:1200] + "\n...")


## Step 3: A minimal agent loop (retrieve → answer)

This is the smallest “agentic retrieval” loop:
- The LLM decides whether to **retrieve** or **answer**.
- If it retrieves, we call `ragnav_retrieve_raw()` and feed results back.
- Then it answers with evidence.

This is intentionally minimal (no orchestration frameworks required).


In [ ]:
from collections.abc import Callable

from ragnav.json_utils import extract_json


RetrieveFn = Callable[[str], list[dict]]


def agentic_answer(query: str, retrieve_fn: RetrieveFn, *, max_steps: int = 3) -> str:
    memory = [
        {
            "role": "system",
            "content": (
                "You are an agent that can call a retrieval tool. "
                "You must ALWAYS output strict JSON with either: "
                '{"action":"retrieve","query":"..."} or {"action":"answer","answer":"..."}. '
                "Prefer retrieving evidence before answering."
            ),
        }
    ]

    tool_spec = {
        "name": "ragnav_retrieve_raw",
        "input": {"query": "string"},
        "output": [
            {
                "doc_id": "string",
                "block_id": "string",
                "title": "string|null",
                "anchors": "object",
                "content": "string",
            }
        ],
    }

    for _ in range(max_steps):
        memory.append(
            {
                "role": "user",
                "content": (
                    f"User query: {query}\n\n"
                    f"Tool available: {json.dumps(tool_spec)}\n\n"
                    "Decide next action."
                ),
            }
        )

        raw = llm.chat(messages=memory, temperature=0)
        try:
            action = extract_json(raw)
        except Exception:
            action = {"action": "retrieve", "query": query}

        if isinstance(action, dict) and action.get("action") == "retrieve":
            q = str(action.get("query") or query)
            hits = retrieve_fn(q)
            memory.append(
                {
                    "role": "user",
                    "content": "Tool result (ragnav_retrieve_raw):\n" + json.dumps(hits, indent=2)[:12000],
                }
            )
            continue

        if isinstance(action, dict) and action.get("action") == "answer":
            return str(action.get("answer") or "").strip()

    memory.append(
        {
            "role": "user",
            "content": "Return final answer JSON now: {\"action\":\"answer\",\"answer\":\"...\"}",
        }
    )
    final_raw = llm.chat(messages=memory, temperature=0)
    final = extract_json(final_raw)
    return str(final.get("answer") if isinstance(final, dict) else final_raw).strip()


query = "What evaluation methods are described in the survey?"
answer = agentic_answer(query, retrieve_fn=lambda q: ragnav_retrieve_raw(q, max_blocks=6))
print_wrapped(answer)


## PDF variant (matches the PageIndex arXiv flow)

PageIndex’s cookbook demonstrates agentic retrieval on a real arXiv PDF like:

```python
pdf_url = "https://arxiv.org/pdf/2507.13334.pdf"
```

Below is the same idea with RAGNav:
- download the PDF
- ingest **per-page** blocks (with `anchors={"page": N}` for provenance)
- run the same agent loop using `retrieve_raw()`


In [ ]:
# Install extras if needed:
# %pip install -e ".[mistral,pdf]"

from ragnav.ingest.pdf import PdfIngestOptions, ingest_pdf_bytes
from ragnav.net import download_pdf

pdf_url = "https://arxiv.org/pdf/2507.13334.pdf"
pdf_bytes = download_pdf(pdf_url)

# Cap pages for cost/latency; increase if you want more context.
pdf_doc, pdf_blocks = ingest_pdf_bytes(
    pdf_bytes,
    name="2507.13334.pdf",
    metadata={"source": "arxiv", "url": pdf_url},
    opts=PdfIngestOptions(max_pages=20),
)

pdf_index = RAGNavIndex.build(documents=[pdf_doc], blocks=pdf_blocks, llm=llm)
pdf_retriever = RAGNavRetriever(index=pdf_index, llm=llm)

print(f"pages_ingested={len(pdf_blocks)}")


In [ ]:
query = "What are the evaluation methods used in this paper?"
answer = agentic_answer(
    query,
    retrieve_fn=lambda q: pdf_retriever.retrieve_raw(q, max_blocks=8),
)
print_wrapped(answer)


## Example output (will vary)

After running the cells above, you should see outputs similar to:

- Step 1 ingestion:

```text
docs=2, blocks=...
```

- Tool output preview (`retrieve_raw`):

```json
[
  {
    "doc_id": "md:survey.md",
    "block_id": "md:survey.md#h0",
    "title": "Context Engineering Survey (Excerpt)",
    "anchors": {"line_start": 2, "line_end": 3},
    "content": "..."
  }
]
```

- Final answer:

```text
The survey describes component-level assessment methods (prompt evaluation, long-context evaluation, self-refinement, structured/relational integration) and benchmarks such as LongMemEval, BFCL/T-Eval, and WebArena.
```


## PDF variant output note

The PDF variant prints `pages_ingested=...` and then an answer. Your exact answer will vary by model/version and the `max_pages` cap.

If you want higher recall, increase `PdfIngestOptions(max_pages=...)` (at the cost of more embedding calls).
